# Auralis — Quantitative Evaluation

Objective evaluation of the AI-Based Cocktail Party Attention System on a LibriSpeech male/female mixture.

**Metrics**
| Metric | Range | Higher is better |
|--------|-------|------------------|
| SI-SDR (dB) | −∞ to +∞ | ✓ |
| PESQ | 1.0 – 4.5 | ✓ |
| STOI | 0 – 1 | ✓ |

**Inputs (from `data/processed/demo/`)**
- `mix.wav` — original F+M mixture (baseline input)
- `target.wav` — ground-truth female voice
- `interferer.wav` — ground-truth male voice
- `output.wav` — system output (separated female voice)

Run `python demo.py` first to generate these files.

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable from notebooks/
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from pesq import pesq
from pystoi import stoi

from src.utils import SAMPLE_RATE, load_audio
from src.dsp.stft import compute_stft
from src.ai.classifier import SpeakerClassifier
from src.ai.attention import AttentionModule
from src.dsp.features import extract_all

DEMO_DIR = repo_root / "data" / "processed" / "demo"
MODEL_PATH = repo_root / "models" / "classifier.joblib"
SR = SAMPLE_RATE  # 16 000 Hz

print(f"Repository root : {repo_root}")
print(f"Demo directory  : {DEMO_DIR}")
print(f"Model path      : {MODEL_PATH}")

## 1. Load audio

In [ ]:
mix,         _ = load_audio(DEMO_DIR / "mix.wav",         sr=SR)
target,      _ = load_audio(DEMO_DIR / "target.wav",      sr=SR)
interferer,  _ = load_audio(DEMO_DIR / "interferer.wav",  sr=SR)
output,      _ = load_audio(DEMO_DIR / "output.wav",      sr=SR)

# Align lengths (STFT boundary can add/remove a few samples)
n = min(len(mix), len(target), len(interferer), len(output))
mix, target, interferer, output = mix[:n], target[:n], interferer[:n], output[:n]

duration = n / SR
print(f"Duration : {duration:.2f} s  |  Samples : {n}  |  SR : {SR} Hz")

## 2. Metric definitions

In [ ]:
def si_sdr(reference: np.ndarray, estimate: np.ndarray) -> float:
    """Scale-invariant signal-to-distortion ratio (dB)."""
    ref = reference - reference.mean()
    est = estimate - estimate.mean()
    alpha = np.dot(ref, est) / (np.dot(ref, ref) + 1e-8)
    projection = alpha * ref
    noise = est - projection
    return 10.0 * np.log10(
        (np.dot(projection, projection) + 1e-8) / (np.dot(noise, noise) + 1e-8)
    )


def compute_metrics(reference: np.ndarray, estimate: np.ndarray, sr: int) -> dict:
    """Return SI-SDR, PESQ (wideband), and STOI for a (reference, estimate) pair."""
    ref_f64 = reference.astype(np.float64)
    est_f64 = estimate.astype(np.float64)
    return {
        "SI-SDR (dB)": round(si_sdr(ref_f64, est_f64), 2),
        "PESQ": round(float(pesq(sr, ref_f64, est_f64, "wb")), 3),
        "STOI": round(float(stoi(ref_f64, est_f64, sr, extended=False)), 3),
    }


print("Metric functions defined.")

## 3. Results: baseline vs. system output

In [ ]:
baseline_metrics = compute_metrics(target, mix, SR)         # unprocessed mix vs target
system_metrics   = compute_metrics(target, output, SR)      # system output vs target

# Pretty-print comparison table
header = f"{'Metric':<14} {'Baseline (mix)':>16} {'System (output)':>16} {'Δ':>10}"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)
for metric in ["SI-SDR (dB)", "PESQ", "STOI"]:
    b = baseline_metrics[metric]
    s = system_metrics[metric]
    delta = s - b
    sign  = "+" if delta >= 0 else ""
    print(f"{metric:<14} {b:>16.3f} {s:>16.3f} {sign}{delta:>9.3f}")
print(sep)

## 4. Waveform comparison

In [ ]:
t = np.linspace(0, duration, n)
signals = [
    ("Target (ground truth)",  target,     "#2ecc71"),
    ("Mix (baseline input)",   mix,        "#e74c3c"),
    ("Output (system)",        output,     "#3498db"),
    ("Interferer (male)",      interferer, "#95a5a6"),
]

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
fig.suptitle("Waveform comparison", fontsize=14, fontweight="bold")

for ax, (label, sig, color) in zip(axes, signals):
    ax.plot(t, sig, color=color, linewidth=0.6, alpha=0.85)
    ax.set_ylabel(label, fontsize=9)
    ax.set_ylim(-1.05, 1.05)
    ax.grid(True, linewidth=0.4, alpha=0.5)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

## 5. Spectrogram comparison

In [ ]:
def log_spectrogram(audio: np.ndarray, sr: int, title: str, ax: plt.Axes) -> None:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=512, hop_length=128)), ref=np.max)
    img = librosa.display.specshow(D, sr=sr, hop_length=128, x_axis="time", y_axis="hz", ax=ax, cmap="magma")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 4000)  # focus on speech range
    return img


fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.suptitle("Log-magnitude spectrograms (0 – 4 kHz)", fontsize=13, fontweight="bold")

panels = [
    ("Mix — baseline input (F + M)",          mix),
    ("Target — ground truth (female only)",   target),
    ("Output — system separation",            output),
    ("Interferer — ground truth (male only)", interferer),
]
for ax, (title, sig) in zip(axes.flat, panels):
    img = log_spectrogram(sig, SR, title, ax)

fig.colorbar(img, ax=list(axes.flat), label="dB", shrink=0.6)
plt.tight_layout()
plt.show()

## 6. Attention mask visualization

In [ ]:
classifier = SpeakerClassifier.load(MODEL_PATH)
attention  = AttentionModule(classifier)
mask       = attention.compute_mask(mix, sr=SR)   # (n_frames,), values in [0, 1]

# Also extract pitch (female range) for overlay
features = extract_all(mix, sr=SR)                 # (44, n_frames)
pitch    = features[39, :]                         # index 39 = female-range F0

frame_times = librosa.frames_to_time(
    np.arange(len(mask)), sr=SR, hop_length=512
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
fig.suptitle("Attention mask & female pitch — input mix", fontsize=13, fontweight="bold")

ax1.fill_between(frame_times, mask, alpha=0.7, color="#3498db", label="Attention weight (F)")
ax1.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Decision boundary")
ax1.set_ylabel("P(female frame)")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9)
ax1.grid(True, linewidth=0.4, alpha=0.5)

ax2.fill_between(frame_times, pitch, alpha=0.7, color="#2ecc71", label="F0 — female range (150–310 Hz)")
ax2.set_ylabel("F0 (Hz)")
ax2.set_xlabel("Time (s)")
ax2.set_ylim(0, 350)
ax2.legend(fontsize=9)
ax2.grid(True, linewidth=0.4, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Mean attention weight : {mask.mean():.3f}")
print(f"Frames above 0.5 (F)  : {(mask > 0.5).sum()} / {len(mask)} ({100*(mask>0.5).mean():.1f}%)")

## 7. Summary

In [ ]:
print("=" * 55)
print("  AURALIS v0.1.x — EVALUATION SUMMARY")
print("=" * 55)
print()
print(f"  Duration : {duration:.2f} s | SR : {SR} Hz")
print()
print(f"  {'Metric':<12} {'Baseline':>10} {'System':>10} {'Δ':>8}")
print(f"  {'-'*44}")
for metric in ["SI-SDR (dB)", "PESQ", "STOI"]:
    b = baseline_metrics[metric]
    s = system_metrics[metric]
    d = s - b
    sign = "+" if d >= 0 else ""
    print(f"  {metric:<12} {b:>10.3f} {s:>10.3f} {sign}{d:>7.3f}")
print()
si_sdr_improvement = system_metrics['SI-SDR (dB)'] - baseline_metrics['SI-SDR (dB)']
if si_sdr_improvement > 2:
    verdict = "PASS — system meaningfully improves separation."
elif si_sdr_improvement > 0:
    verdict = "MARGINAL — small positive improvement over baseline."
else:
    verdict = "FAIL — system does not outperform unprocessed mix."
print(f"  Verdict: {verdict}")
print("=" * 55)